# Démo — Optimiseur de coûts opérationnels miniers

Ce notebook illustre l'architecture du projet de bout en bout, sur la chaîne
opérationnelle **forage -> sautage -> chargement -> transport -> énergie** :
1. Données synthétiques (KPI quotidiens + flotte d'équipements)
2. Modèle de coûts (décomposition par étape, tendance)
3. Simulation Monte Carlo (risque de dépassement de budget / d'objectif de tonnage)
4. Optimisation (recherche opérationnelle) — arbitrage flotte propre / sous-traitance

> ⚠️ Toutes les données utilisées ici sont **100% synthétiques**, générées par `data/generate_synthetic_data.py`. Aucune donnée réelle n'est utilisée.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.cost_model import load_daily_kpi, cost_breakdown, cost_trend, month_over_month_delta, summary_kpis
from src.simulation import run_monte_carlo
from src.optimization import run_scenario

## 1. KPI quotidiens (production, coûts par étape, disponibilité, HSE)

In [ ]:
kpi = load_daily_kpi()
kpi.head()

In [ ]:
kpi.set_index("date")["cout_total_usd_t"].plot(title="Coût total (USD/tonne) — historique simulé", figsize=(10, 3));

## 2. Modèle de coûts — décomposition par étape

In [ ]:
breakdown = cost_breakdown(kpi, period="last_90d")
print(breakdown)
breakdown.plot.bar(title="Coût moyen par étape, 90 derniers jours (USD/tonne)", figsize=(7, 3));

In [ ]:
print(summary_kpis(kpi))
print(month_over_month_delta(kpi))

## 3. Simulation Monte Carlo

Ré-échantillonnage (bootstrap) de journées historiques pour simuler des milliers de scénarios plausibles du mois à venir, et quantifier le risque de dépasser un budget coût ou de manquer un objectif de tonnage — sans supposer de loi de probabilité théorique.

In [ ]:
recent_avg_cost = kpi["cout_total_usd_t"].tail(90).mean()
recent_avg_tonnage_month = kpi["tonnes_produites"].tail(90).mean() * 30

sim = run_monte_carlo(
    horizon_days=30,
    cost_budget_usd_t=round(recent_avg_cost * 1.02, 1),
    tonnage_target=round(recent_avg_tonnage_month * 0.95),
)
print(f"Coût simulé (USD/t) : P10={sim.cout_p10_usd_t}  P50={sim.cout_p50_usd_t}  P90={sim.cout_p90_usd_t}")
print(f"Tonnage simulé (30j) : P10={sim.tonnage_p10:,}  P50={sim.tonnage_p50:,}  P90={sim.tonnage_p90:,}")
print(f"P(coût > budget) = {sim.proba_depassement_budget:.1%}")
print(f"P(tonnage < objectif) = {sim.proba_sous_objectif:.1%}")

## 4. Optimisation — arbitrage flotte propre / sous-traitance

Programme linéaire (scipy.optimize.linprog) : pour un objectif de tonnage sur 7 jours, trouve la répartition d'heures la moins coûteuse entre flotte propre (capacité limitée) et sous-traitance (coût plus élevé, capacité non limitée), par étape.

In [ ]:
for mult in [1.0, 1.10, 1.25]:
    scenario = run_scenario(target_multiplier=mult)
    print(f"--- Objectif = {int(mult*100)}% de la production récente ({scenario.target_tonnage} t / 7j) ---")
    print(f"Coût total optimisé : {scenario.total_cost_usd:,} USD")
    print(f"Manque à gagner (flotte propre seule) : {scenario.baseline_shortfall_tonnes:,} t")
    print(f"Coût marginal de la sous-traitance : {scenario.cout_marginal_usd_par_tonne} USD/t")
    print()

In [ ]:
scenario = run_scenario(target_multiplier=1.10)
scenario.stage_allocation

## 5. Limites et pistes d'amélioration

- **Données et débits stylisés** : les débits horaires par étape (tonnes/heure) sont calibrés pour que la capacité de flotte simulée corresponde à la production moyenne du jeu de données synthétique — ce ne sont pas des ratios d'ingénierie minière réels.
- **Sautage hors optimisation de flotte** : traité comme un coût variable au tonnage, car il ne repose pas sur une flotte d'équipement avec heures d'exploitation dans ce modèle.
- **Simulation par bootstrap historique** : simple et robuste, mais suppose que les 180 derniers jours restent représentatifs du futur proche ; une version plus avancée pondérerait les tirages ou modéliserait explicitement une tendance/saisonnalité.
- **Optimisation** : modèle statique par horizon (7 jours) ; une version plus avancée intégrerait la variabilité stochastique de la disponibilité flotte (optimisation robuste / sous incertitude) plutôt que des capacités moyennes historiques — un pont naturel avec le module de simulation.